# COMP-4740: GNN-based Household Object Recognition via Superpixel Graphs
**Author:** Harmit Patel | University of Windsor | Winter 2026

**Models:** Baseline GNN (GraphSAGE) vs GCN vs GAT

---
Run cells **in order**. After Cell 2 finishes: **Runtime -> Restart session**, then continue from Cell 3.

In [ ]:
# CELL 1: Check PyTorch version
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print('Run Cell 2 next.')

In [ ]:
# CELL 2: Install all libraries
# After this finishes -> Runtime -> Restart session
!pip install torch-geometric --quiet
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.5.0+cu121.html --quiet
!pip install scikit-image scikit-learn --quiet
!pip install networkx matplotlib seaborn pandas --quiet
print('Done! Now go to Runtime -> Restart session, then run Cell 3.')

In [ ]:
# CELL 3: Verify imports (run AFTER runtime restart)
import torch
import torch_geometric
import skimage
import networkx as nx
import matplotlib.pyplot as plt
import sklearn
import numpy as np
import pandas as pd
print(f'PyTorch: {torch.__version__}')
print(f'PyTorch Geometric: {torch_geometric.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('All good! Ready to build.')

In [ ]:
# CELL 4: Download CIFAR-10 dataset
import torchvision
import torchvision.transforms as transforms
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])
print('Downloading CIFAR-10...')
train_dataset = torchvision.datasets.CIFAR10(root='/content/data', train=True, download=True, transform=transform)
test_dataset  = torchvision.datasets.CIFAR10(root='/content/data', train=False, download=True, transform=transform)
print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')
print(f'Classes: {train_dataset.classes}')

In [ ]:
# CELL 5: Visualize sample images
import matplotlib.pyplot as plt
from skimage.util import img_as_float
classes = train_dataset.classes
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('CIFAR-10 Dataset - One Sample per Class', fontsize=14, fontweight='bold')
for i, ax in enumerate(axes.flat):
    for j in range(len(train_dataset)):
        img, label = train_dataset[j]
        if label == i:
            ax.imshow(img.permute(1,2,0).numpy())
            ax.set_title(classes[i], fontsize=11)
            ax.axis('off')
            break
plt.tight_layout()
plt.savefig('/content/dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dataset_samples.png')

In [ ]:
# CELL 6: Superpixel graph conversion function
import torch
import numpy as np
from skimage.segmentation import slic
from skimage.util import img_as_float
from torch_geometric.data import Data
import warnings
warnings.filterwarnings('ignore')

def image_to_graph(image_tensor, n_segments=75, compactness=10):
    image_np = image_tensor.permute(1, 2, 0).numpy()
    image_np = img_as_float(image_np)
    H, W, C = image_np.shape
    segments = slic(image_np, n_segments=n_segments, compactness=compactness, start_label=0)
    num_nodes = segments.max() + 1
    node_features = []
    for seg_id in range(num_nodes):
        mask = (segments == seg_id)
        if mask.sum() == 0:
            node_features.append(np.zeros(5))
            continue
        mean_color = image_np[mask].mean(axis=0)
        coords = np.argwhere(mask)
        mean_y = coords[:, 0].mean() / H
        mean_x = coords[:, 1].mean() / W
        node_features.append(np.concatenate([mean_color, [mean_x, mean_y]]))
    node_features = torch.tensor(np.array(node_features), dtype=torch.float)
    edges = set()
    for y in range(H):
        for x in range(W):
            curr = segments[y, x]
            if x + 1 < W:
                nb = segments[y, x + 1]
                if curr != nb:
                    edges.add((min(curr,nb), max(curr,nb)))
            if y + 1 < H:
                nb = segments[y + 1, x]
                if curr != nb:
                    edges.add((min(curr,nb), max(curr,nb)))
    if len(edges) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    else:
        el = list(edges)
        src = [e[0] for e in el] + [e[1] for e in el]
        dst = [e[1] for e in el] + [e[0] for e in el]
        edge_index = torch.tensor([src, dst], dtype=torch.long)
    return Data(x=node_features, edge_index=edge_index)

sample_img, _ = train_dataset[0]
g = image_to_graph(sample_img)
print(f'Graph function ready!')
print(f'Nodes: {g.num_nodes}, Edges: {g.num_edges}, Features per node: {g.x.shape[1]}')

In [ ]:
# CELL 7: Visualize superpixel graphs (figures for the paper)
import matplotlib.pyplot as plt
import numpy as np
from skimage.segmentation import slic, mark_boundaries
from skimage.util import img_as_float

classes = train_dataset.classes
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('SLIC Superpixel Segmentation Across All Classes', fontsize=14, fontweight='bold')
for i in range(10):
    for j in range(len(train_dataset)):
        img_j, lbl_j = train_dataset[j]
        if lbl_j == i:
            img_np = img_as_float(img_j.permute(1,2,0).numpy())
            segs = slic(img_np, n_segments=75, compactness=10, start_label=0)
            axes.flat[i].imshow(mark_boundaries(img_np, segs))
            axes.flat[i].set_title(f'{classes[i]} ({segs.max()+1} nodes)')
            axes.flat[i].axis('off')
            break
plt.tight_layout()
plt.savefig('/content/superpixel_grid.png', dpi=150, bbox_inches='tight')
plt.show()

img0, lbl0 = train_dataset[0]
img_np0 = img_as_float(img0.permute(1,2,0).numpy())
segs0 = slic(img_np0, n_segments=75, compactness=10, start_label=0)
fig2, ax2 = plt.subplots(1, 2, figsize=(10, 4))
fig2.suptitle(f'Superpixel Graph Example - Class: {classes[lbl0]}', fontweight='bold')
ax2[0].imshow(img_np0); ax2[0].set_title('Original'); ax2[0].axis('off')
ax2[1].imshow(mark_boundaries(img_np0, segs0)); ax2[1].set_title(f'SLIC ({segs0.max()+1} superpixels)'); ax2[1].axis('off')
plt.tight_layout()
plt.savefig('/content/graph_example.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: superpixel_grid.png and graph_example.png')

In [ ]:
# CELL 8: Convert images to graphs (~8-10 minutes)
import random
from torch_geometric.loader import DataLoader as PyGDataLoader

def build_graph_dataset(dataset, max_samples=2000, n_segments=75):
    graphs = []
    indices = list(range(len(dataset)))
    random.shuffle(indices)
    indices = indices[:max_samples]
    for i, idx in enumerate(indices):
        img, label = dataset[idx]
        try:
            graph = image_to_graph(img, n_segments=n_segments)
            graph.y = torch.tensor([label], dtype=torch.long)
            graphs.append(graph)
        except:
            continue
        if (i + 1) % 250 == 0:
            print(f'  Converted {i+1}/{max_samples}...')
    return graphs

random.seed(42)
print('Converting train images to graphs...')
train_graphs = build_graph_dataset(train_dataset, max_samples=2000)
print(f'Train graphs: {len(train_graphs)}')
print('Converting test images to graphs...')
test_graphs = build_graph_dataset(test_dataset, max_samples=500)
print(f'Test graphs: {len(test_graphs)}')
train_loader = PyGDataLoader(train_graphs, batch_size=32, shuffle=True)
test_loader  = PyGDataLoader(test_graphs,  batch_size=32, shuffle=False)
print('DataLoaders ready!')

In [ ]:
# CELL 9: Define all 3 GNN models
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, global_mean_pool

class BaselineGNN(nn.Module):
    def __init__(self, in_ch, hid, out_ch):
        super().__init__()
        self.conv1 = SAGEConv(in_ch, hid)
        self.conv2 = SAGEConv(hid, hid)
        self.conv3 = SAGEConv(hid, hid)
        self.cls = nn.Sequential(nn.Linear(hid, hid//2), nn.ReLU(), nn.Dropout(0.5), nn.Linear(hid//2, out_ch))
    def forward(self, x, edge_index, batch):
        x = F.dropout(F.relu(self.conv1(x, edge_index)), 0.3, self.training)
        x = F.dropout(F.relu(self.conv2(x, edge_index)), 0.3, self.training)
        x = F.relu(self.conv3(x, edge_index))
        return self.cls(global_mean_pool(x, batch))

class GCN(nn.Module):
    def __init__(self, in_ch, hid, out_ch):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hid)
        self.conv2 = GCNConv(hid, hid)
        self.conv3 = GCNConv(hid, hid)
        self.bn1 = nn.BatchNorm1d(hid)
        self.bn2 = nn.BatchNorm1d(hid)
        self.cls = nn.Sequential(nn.Linear(hid, hid//2), nn.ReLU(), nn.Dropout(0.5), nn.Linear(hid//2, out_ch))
    def forward(self, x, edge_index, batch):
        x = F.dropout(F.relu(self.bn1(self.conv1(x, edge_index))), 0.3, self.training)
        x = F.dropout(F.relu(self.bn2(self.conv2(x, edge_index))), 0.3, self.training)
        x = F.relu(self.conv3(x, edge_index))
        return self.cls(global_mean_pool(x, batch))

class GAT(nn.Module):
    def __init__(self, in_ch, hid, out_ch, heads=4):
        super().__init__()
        self.conv1 = GATConv(in_ch, hid, heads=heads, dropout=0.3)
        self.conv2 = GATConv(hid*heads, hid, heads=heads, dropout=0.3)
        self.conv3 = GATConv(hid*heads, hid, heads=1, dropout=0.3)
        self.cls = nn.Sequential(nn.Linear(hid, hid//2), nn.ReLU(), nn.Dropout(0.5), nn.Linear(hid//2, out_ch))
    def forward(self, x, edge_index, batch):
        x = F.dropout(F.elu(self.conv1(x, edge_index)), 0.3, self.training)
        x = F.dropout(F.elu(self.conv2(x, edge_index)), 0.3, self.training)
        x = F.elu(self.conv3(x, edge_index))
        return self.cls(global_mean_pool(x, batch))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
IN, HID, OUT = 5, 128, 10
baseline_model = BaselineGNN(IN, HID, OUT).to(device)
gcn_model      = GCN(IN, HID, OUT).to(device)
gat_model      = GAT(IN, HID, OUT).to(device)
def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'Baseline GNN params: {count_params(baseline_model):,}')
print(f'GCN params:          {count_params(gcn_model):,}')
print(f'GAT params:          {count_params(gat_model):,}')

In [ ]:
# CELL 10: Train and eval helper functions
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y.squeeze())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pred = out.argmax(dim=1)
        correct += (pred == batch.y.squeeze()).sum().item()
        total   += batch.y.size(0)
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = criterion(out, batch.y.squeeze())
            total_loss += loss.item()
            pred = out.argmax(dim=1)
            correct += (pred == batch.y.squeeze()).sum().item()
            total   += batch.y.size(0)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(batch.y.squeeze().cpu().numpy())
    return total_loss / len(loader), correct / total, all_preds, all_labels

print('Training functions ready!')

In [ ]:
# CELL 11: Train all 3 models (~35-45 mins on T4 GPU)
import time
EPOCHS = 30
criterion = nn.CrossEntropyLoss()
results = {}

def train_model(model, name, train_loader, test_loader, epochs=EPOCHS):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    tr_losses, te_losses, tr_accs, te_accs = [], [], [], []
    print(f'\nTraining: {name}')
    print('-' * 50)
    start = time.time()
    for epoch in range(1, epochs+1):
        trl, tra = train_epoch(model, train_loader, optimizer, criterion, device)
        tel, tea, _, _ = evaluate(model, test_loader, criterion, device)
        scheduler.step()
        tr_losses.append(trl); te_losses.append(tel)
        tr_accs.append(tra);   te_accs.append(tea)
        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:2d}/{epochs} | Train Acc: {tra:.4f} | Test Acc: {tea:.4f}')
    elapsed = time.time() - start
    _, final_acc, preds, labels = evaluate(model, test_loader, criterion, device)
    print(f'  Final Test Accuracy: {final_acc*100:.2f}%  |  Time: {elapsed:.1f}s')
    return {'train_losses': tr_losses, 'test_losses': te_losses,
            'train_accs': tr_accs, 'test_accs': te_accs,
            'final_acc': final_acc, 'time': elapsed,
            'preds': preds, 'labels': labels}

results['Baseline GNN'] = train_model(baseline_model, 'Baseline GNN', train_loader, test_loader)
results['GCN']          = train_model(gcn_model,      'GCN',          train_loader, test_loader)
results['GAT']          = train_model(gat_model,      'GAT',          train_loader, test_loader)

print('\n=== FINAL SUMMARY ===')
for name, res in results.items():
    print(f'  {name:<15} Test Acc: {res["final_acc"]*100:.2f}%  Time: {res["time"]:.1f}s')

In [ ]:
# CELL 12: Result plots for the paper
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

colors = {'Baseline GNN': '#2196F3', 'GCN': '#4CAF50', 'GAT': '#FF5722'}
classes = train_dataset.classes

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('GNN Model Comparison - Superpixel Graph Classification\nHarmit Patel | COMP-4740 | University of Windsor', fontsize=13, fontweight='bold')

for name, res in results.items():
    axes[0,0].plot(res['train_accs'], label=name, color=colors[name], linewidth=2)
axes[0,0].set_title('Training Accuracy', fontweight='bold')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Accuracy')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

for name, res in results.items():
    axes[0,1].plot(res['test_accs'], label=name, color=colors[name], linewidth=2)
axes[0,1].set_title('Test Accuracy', fontweight='bold')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Accuracy')
axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

model_names = list(results.keys())
final_accs  = [results[n]['final_acc']*100 for n in model_names]
bars = axes[1,0].bar(model_names, final_accs, color=[colors[n] for n in model_names], edgecolor='black')
for bar, acc in zip(bars, final_accs):
    axes[1,0].text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.3, f'{acc:.2f}%', ha='center', fontweight='bold')
axes[1,0].set_title('Final Test Accuracy', fontweight='bold')
axes[1,0].set_ylabel('Accuracy (%)')
axes[1,0].set_ylim(0, max(final_accs)+10)
axes[1,0].grid(True, alpha=0.3, axis='y')

for name, res in results.items():
    axes[1,1].plot(res['test_losses'], label=name, color=colors[name], linewidth=2)
axes[1,1].set_title('Test Loss', fontweight='bold')
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Loss')
axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/results_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best_name = max(results, key=lambda n: results[n]['final_acc'])
fig2, ax2 = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(results[best_name]['labels'], results[best_name]['preds'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=ax2)
ax2.set_title(f'Confusion Matrix - {best_name} (Best Model)', fontweight='bold')
ax2.set_ylabel('True Label'); ax2.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best model: {best_name} - {results[best_name]["final_acc"]*100:.2f}%')
print('Saved: results_comparison.png, confusion_matrix.png')

In [ ]:
# CELL 13: Print full results table
from sklearn.metrics import classification_report
import pandas as pd

print('=' * 60)
print('COMP-4740 FINAL RESULTS TABLE')
print('GNN-based Object Recognition via Superpixel Graphs')
print('Harmit Patel | University of Windsor | 2026')
print('=' * 60)
rows = []
for name, res in results.items():
    rows.append({
        'Model': name,
        'Test Acc (%)': f'{res["final_acc"]*100:.2f}',
        'Train Acc (%)': f'{res["train_accs"][-1]*100:.2f}',
        'Test Loss': f'{res["test_losses"][-1]:.4f}',
        'Time (s)': f'{res["time"]:.1f}'
    })
print(pd.DataFrame(rows).to_string(index=False))
best_name = max(results, key=lambda n: results[n]['final_acc'])
print(f'\nClassification Report - {best_name}:')
print(classification_report(results[best_name]['labels'], results[best_name]['preds'], target_names=train_dataset.classes))

In [ ]:
# CELL 14: Save all models and results
import pickle
torch.save(baseline_model.state_dict(), '/content/baseline_gnn.pth')
torch.save(gcn_model.state_dict(),      '/content/gcn_model.pth')
torch.save(gat_model.state_dict(),      '/content/gat_model.pth')
with open('/content/results.pkl', 'wb') as f:
    pickle.dump(results, f)
print('All files saved to /content/:')
print('  Models: baseline_gnn.pth, gcn_model.pth, gat_model.pth')
print('  Plots:  results_comparison.png, confusion_matrix.png')
print('  Plots:  superpixel_grid.png, graph_example.png, dataset_samples.png')
print('  Data:   results.pkl')
print()
print('Download: Files panel (left sidebar) -> right-click -> Download')
print()
print('PROJECT COMPLETE!')